In [25]:
import pandas as pd
import numpy as np
import os

In [26]:
input_week_1_dir = "/Users/akshayprabhu/Desktop/Kaggle NFL/data/train/input_2023_w01.csv"
input_df = pd.read_csv(input_week_1_dir)

In [27]:
input_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285714 entries, 0 to 285713
Data columns (total 23 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   game_id                   285714 non-null  int64  
 1   play_id                   285714 non-null  int64  
 2   player_to_predict         285714 non-null  bool   
 3   nfl_id                    285714 non-null  int64  
 4   frame_id                  285714 non-null  int64  
 5   play_direction            285714 non-null  object 
 6   absolute_yardline_number  285714 non-null  int64  
 7   player_name               285714 non-null  object 
 8   player_height             285714 non-null  object 
 9   player_weight             285714 non-null  int64  
 10  player_birth_date         285714 non-null  object 
 11  player_position           285714 non-null  object 
 12  player_side               285714 non-null  object 
 13  player_role               285714 non-null  o

In [28]:
input_df.head(5)

,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,...,player_role,x,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y
0,2023090700,101,False,54527,1,right,42,Bryan Cook,6-1,210,...,Defensive Coverage,52.33,36.94,0.09,0.39,322.40,238.24,21,63.259998,-0.22
1,2023090700,101,False,54527,2,right,42,Bryan Cook,6-1,210,...,Defensive Coverage,52.33,36.94,0.04,0.61,200.89,236.05,21,63.259998,-0.22
2,2023090700,101,False,54527,3,right,42,Bryan Cook,6-1,210,...,Defensive Coverage,52.33,36.93,0.12,0.73,147.55,240.60,21,63.259998,-0.22
3,2023090700,101,False,54527,4,right,42,Bryan Cook,6-1,210,...,Defensive Coverage,52.35,36.92,0.23,0.81,131.40,244.25,21,63.259998,-0.22
4,2023090700,101,False,54527,5,right,42,Bryan Cook,6-1,210,...,Defensive Coverage,52.37,36.90,0.35,0.82,123.26,244.25,21,63.259998,-0.22


### Input Notes

1. player_to_predict
    - boolean that marks whether that players movement is what is needed to be predicted
    - one offensive player (reciever) per play
    - variable amount of defensive players per play

2. play direction
    - offense and defense is not standardized
    - some plays the offense faces right while others it may face left
    
3. x and y
    - coordinates on the field

4. s and a
    - speed and acceleration

5. dir and o
    - direction and orientation, angles

6. num_frames_output
    - number of output frames that need to be predicted for the play

7. ball land
    - x and y coordinates that mark where the ball lands
    - static for each play

### Output

The output data frame only gives us the x and y coordinates from each player where the player_to_predict category is labeled as true. The info is show below:

In [29]:
output_week_1_dir = "/Users/akshayprabhu/Desktop/Kaggle NFL/data/train/output_2023_w01.csv"
output_df = pd.read_csv(output_week_1_dir)

In [30]:
output_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32088 entries, 0 to 32087
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   game_id   32088 non-null  int64  
 1   play_id   32088 non-null  int64  
 2   nfl_id    32088 non-null  int64  
 3   frame_id  32088 non-null  int64  
 4   x         32088 non-null  float64
 5   y         32088 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 1.5 MB


In [31]:
output_df.head(5)

,game_id,play_id,nfl_id,frame_id,x,y
0,2023090700,101,46137,1,56.22,17.28
1,2023090700,101,46137,2,56.63,16.88
2,2023090700,101,46137,3,57.06,16.46
3,2023090700,101,46137,4,57.48,16.02
4,2023090700,101,46137,5,57.91,15.56


In [32]:
import matplotlib.pyplot as plt
import plotly.express as px
import matplotlib.animation as animation

In [33]:
# Create the field (120 x 53.3 yards in NFL Next Gen Stats coords)
def draw_field(ax=None):
    if ax is None:
        ax = plt.gca()
    # Field outline
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 53.3)
    ax.set_facecolor("mediumseagreen")
    # End zones
    ax.add_patch(plt.Rectangle((0, 0), 10, 53.3, color="lightgrey"))
    ax.add_patch(plt.Rectangle((110, 0), 10, 53.3, color="lightgrey"))
    
    for x in range(10, 111, 5):
        if x % 10 == 0:  
            # Solid line every 10 yards
            ax.plot([x, x], [0, 53.3], color="white", linewidth=2)
        else:
            # Dashed line every 5 yards
            ax.plot([x, x], [0, 53.3], color="white", linewidth=1, linestyle="--")
            
    # Hash marks: every yard line from 1 to 99
    for x in range(10, 111):
        # Left side hash
        ax.plot([x, x], [23.36-0.4, 23.36+0.4], color="white", linewidth=2)
        # Right side hash
        ax.plot([x, x], [29.94-0.4, 29.94+0.4], color="white", linewidth=2)
    
    # Yard numbers every 10 yards (don’t label inside endzones)
    for x in range(20, 110, 10):
        num = x - 10 if x <= 50 else 110 - x  # Flip numbers after midfield
        ax.text(x, 53.3 - 2, str(num), color="white", ha="center", va="center", fontsize=14, weight="bold")
        ax.text(x, 2, str(num), color="white", ha="center", va="center", fontsize=14, weight="bold", rotation=180)
        
    return ax

In [34]:
## game_id = 2023091010
## play_id = 4426

In [59]:
i_play = input_df[(input_df['game_id'] == 2023091010) & (input_df['play_id'] == 4426)]

In [60]:
o_play = output_df[(output_df['game_id'] == 2023091010) & (output_df['play_id'] == 4426)]

In [61]:
# Step 1: get the max input frame per play
frame_offsets = (
    i_play.groupby(["game_id", "play_id"])["frame_id"]
    .max()
    .reset_index()
    .rename(columns={"frame_id": "maxInputFrame"})
)

# Step 2: merge offsets into output_df
output_shifted = o_play.merge(frame_offsets, on=["game_id", "play_id"], how="left")

# Step 3: shift the output frameIds
output_shifted["frame_id"] = output_shifted["frame_id"] + output_shifted["maxInputFrame"]

# Step 4: (optional) bring over team/position info from input
output_shifted = output_shifted.merge(
    i_play[["game_id", "play_id", "nfl_id"]].drop_duplicates(),
    on=["game_id", "play_id", "nfl_id"],
    how="left"
)

# Step 5: combine input + adjusted output
play = pd.concat([i_play, output_shifted.drop(columns=["maxInputFrame"])], ignore_index=True)

In [62]:
columns = ['game_id', 'play_id', 'nfl_id', 'frame_id',
          'x', 'y', 'ball_land_x', 'ball_land_y', 'player_side']
play_info = play
play = play[columns]

In [63]:
play.loc[: , 'ball_land_x'] = play['ball_land_x'].fillna(play['ball_land_x'].iloc[1])

In [64]:
play.loc[:, 'ball_land_y'] = play['ball_land_y'].fillna(play['ball_land_y'].iloc[1])

In [65]:
play.loc[:, 'player_side'] = (
    play.groupby('nfl_id')['player_side']
      .transform(lambda x: x.ffill().bfill())
)

In [66]:
play

,game_id,play_id,nfl_id,frame_id,x,y,ball_land_x,ball_land_y,player_side
0,2023091010,4426,52594,1,49.23,34.18,78.260002,8.42,Defense
1,2023091010,4426,52594,2,49.32,34.37,78.260002,8.42,Defense
2,2023091010,4426,52594,3,49.40,34.55,78.260002,8.42,Defense
3,2023091010,4426,52594,4,49.46,34.69,78.260002,8.42,Defense
4,2023091010,4426,52594,5,49.51,34.79,78.260002,8.42,Defense
...,...,...,...,...,...,...,...,...,...
605,2023091010,4426,43454,60,74.59,7.88,78.260002,8.42,Offense
606,2023091010,4426,43454,61,75.56,8.01,78.260002,8.42,Offense
607,2023091010,4426,43454,62,76.49,8.15,78.260002,8.42,Offense
608,2023091010,4426,43454,63,77.30,8.29,78.260002,8.42,Offense


In [67]:
from IPython.display import HTML

In [69]:
def update(frame):
    ax.clear()
    draw_field(ax)
    #frame_data = play[play["frame_id"] == frame]
    
    frame_data = play[play["frame_id"] <= frame]
    frame_data = (
        frame_data
        .sort_values("frame_id")
        .groupby("nfl_id")
        .tail(1)
    )

    off_scatter = ax.scatter(frame_data[frame_data["player_side"] == "Offense"]["x"],
               frame_data[frame_data["player_side"] == "Offense"]["y"],
               c="blue", s=100, label="Offense", zorder=5)

    def_scatter = ax.scatter(frame_data[frame_data["player_side"] == "Defense"]["x"],
               frame_data[frame_data["player_side"] == "Defense"]["y"],
               c="red", s=100, label="Defense", zorder=5)

    
    # Ball landing spot (same for all frames, optional marker)
    play_meta = play.dropna(subset=["ball_land_x", "ball_land_y"]).head(1)
    if not play_meta.empty:
        ax.scatter(play_meta["ball_land_x"], play_meta["ball_land_y"],
                   c="yellow", s=120, marker="*", label="Ball Landing", zorder=6)

    ax.set_title(f"Frame {frame}")
    ax.legend(loc="upper right")

    # IMPORTANT: return artists for FuncAnimation
    return ax.collections

fig, ax = plt.subplots(figsize=(12, 6))

# --- Animate ---
ani = animation.FuncAnimation(
    fig, update,
    frames=sorted(play["frame_id"].unique()),
    interval=100,
    blit=False  # keep False since we redraw the whole field
)
plt.close(fig)
HTML(ani.to_jshtml())
